# VTOL Controller Diagnostics

This notebook turns raw IFC VTOL logs into a readable **controller health report**.

## What You Get
- Event timeline: first airborne, first level-control, first saturation, first upset
- Attitude/rate plots with stage markers (`FF_ONLY`, `P_ONLY`, `PD`, `PID`)
- PID term breakdown (`P`, `I`, `D`, unsaturated output, post-cap output)
- Allocator and per-engine limiter behavior
- Recent run comparison aligned by airborne time

## Quick Start
1. Run all cells top-to-bottom.
2. Check the **Event Timeline** table first.
3. Look at the **Attitude / Rates / Commands** panel to see where divergence starts.
4. Use the **PID Terms** panel to see whether `P`, `I`, or `D` is driving the rail.
5. Use **Allocator / Limiters** to verify whether authority is constrained post-controller.


## Signal Primer

- `pitch_deg`, `bank_deg`: measured attitude angles (deg).
- `pitch_rate_degs`, `roll_rate_degs`: measured angular rates (deg/s).
- `pitch_cmd`, `roll_cmd`: commanded attitude corrections after controller logic.
- `vtol_cmd_*_precap`: command before hard cap.
- `vtol_cmd_*_postcap`: command after cap.
- `vtol_cmd_*_postupset`: command after upset override.
- `alloc_alpha`: allocator differential scaling actually applied.
- `vtol_clamp_high_n`, `vtol_clamp_low_n`: number of engines pinned at bounds.

### Core Control Targets (Setpoints)
- **Attitude setpoints (auto-level mode):** `pitch_target_deg = 0`, `bank_target_deg = 0`.
- **Rate setpoints (damping):** `pitch_rate_target = 0`, `roll_rate_target = 0`.
- **Vertical speed setpoint:** `vs_cmd_ms` (from throttle mapping or altitude-hold path).

### Logged Error Terms
- `vtol_pitch_err` is the pitch-loop error in command sign convention.
- `vtol_roll_err` is the roll-loop error in command sign convention.
- In level-hold segments, these generally correspond to zero-attitude targeting.

Interpretation rule of thumb:
- If commands rail but `alloc_alpha=1` and clamp counts are low, instability is mostly control-law/gain structure.
- If commands rail and clamp counts are high and/or `alloc_alpha<1`, authority limits are dominating.


In [ ]:
from pathlib import Path
import io
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

def resolve_logs_dir(start: Path):
    candidates = []

    # Common notebook launch locations.
    candidates.append((start / '../../logs').resolve())
    candidates.append((start / 'logs').resolve())
    candidates.append((start / 'Integrated Flight Computer/logs').resolve())

    # Walk upward from CWD and look for Integrated Flight Computer/logs.
    for parent in [start] + list(start.parents):
        candidates.append((parent / 'Integrated Flight Computer/logs').resolve())
        candidates.append((parent / 'logs').resolve())

    # De-duplicate while preserving order.
    dedup = []
    seen = set()
    for c in candidates:
        s = str(c)
        if s not in seen:
            seen.add(s)
            dedup.append(c)

    for c in dedup:
        if c.exists() and any(c.glob('ifc_log_*.csv')):
            return c, dedup

    return None, dedup

CWD = Path.cwd()
LOGS_DIR, SEARCHED_DIRS = resolve_logs_dir(CWD)

print(f"Notebook CWD: {CWD}")
if LOGS_DIR is None:
    print('Searched directories:')
    for d in SEARCHED_DIRS:
        print('-', d)
    raise FileNotFoundError('Could not locate a logs directory containing ifc_log_*.csv')

print(f"Using logs dir: {LOGS_DIR}")


## Control System Diagram

This block diagram maps the major VTOL control paths used in IFC logs:
- **Vertical path**: pilot throttle -> VS hold + feed-forward -> collective.
- **Attitude path**: attitude/rate error -> PID -> roll/pitch commands.
- **Allocation path**: collective + commands -> mixer/allocator -> per-engine limits.
- **Plant/feedback path**: engine limits -> vehicle dynamics -> sensors -> controller.
- **Supervisory path**: upset/guards can override/cap commands.

> If you see syntax errors in this section, reload the notebook from disk and re-run all cells.


In [ ]:
import matplotlib.patches as patches

DIAGRAM_LABELS = {
    'pilot': "Pilot\nThrottle",
    'vs_hold': "VS Hold +\nFF Engine Model",
    'collective': "Collective\nCommand",
    'pid': "Attitude/Rate\nPID (Roll/Pitch)",
    'cmds': "Roll/Pitch\nCommands",
    'allocator': "Mixer + Allocator\n(diff scale, caps,\nclamps, alpha)",
    'limits': "Per-Engine\nLimiter Outputs",
    'plant': "Vehicle Dynamics\n(airframe + engines)",
    'sensors': "Sensors/Telemetry\n(angle, rate, VS, AGL)",
    'supervisor': "Upset / Guard\nSupervisor",
}

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

def block(x, y, w, h, label, fc='#f5f7fb', ec='#3b4a66', fs=10):
    rect = patches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle='round,pad=0.01,rounding_size=0.01',
        linewidth=1.4,
        edgecolor=ec,
        facecolor=fc
    )
    ax.add_patch(rect)
    ax.text(x + w / 2, y + h / 2, label, ha='center', va='center', fontsize=fs)
    return rect

def arrow(x1, y1, x2, y2, text=None, color='#2f3e55'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', lw=1.5, color=color))
    if text:
        ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.02, text, ha='center', va='center', fontsize=9, color=color)

# Main blocks
block(0.03, 0.62, 0.12, 0.12, DIAGRAM_LABELS['pilot'])
block(0.20, 0.62, 0.16, 0.12, DIAGRAM_LABELS['vs_hold'])
block(0.40, 0.62, 0.12, 0.12, DIAGRAM_LABELS['collective'])

block(0.20, 0.28, 0.18, 0.14, DIAGRAM_LABELS['pid'])
block(0.42, 0.28, 0.12, 0.14, DIAGRAM_LABELS['cmds'])

block(0.58, 0.48, 0.18, 0.16, DIAGRAM_LABELS['allocator'])
block(0.80, 0.50, 0.16, 0.14, DIAGRAM_LABELS['limits'])
block(0.80, 0.25, 0.16, 0.16, DIAGRAM_LABELS['plant'])
block(0.58, 0.08, 0.18, 0.14, DIAGRAM_LABELS['sensors'])

block(0.03, 0.20, 0.12, 0.12, DIAGRAM_LABELS['supervisor'], fc='#fff4f2', ec='#8b3a2e')

# Connections: vertical path
arrow(0.15, 0.68, 0.20, 0.68, 'thr input')
arrow(0.36, 0.68, 0.40, 0.68, 'collective')
arrow(0.52, 0.68, 0.58, 0.58)

# Connections: attitude path
arrow(0.38, 0.35, 0.42, 0.35, 'cmd')
arrow(0.54, 0.35, 0.58, 0.54)

# Allocation -> plant
arrow(0.76, 0.56, 0.80, 0.56, 'engine limits')
arrow(0.88, 0.50, 0.88, 0.41)

# Plant feedback
arrow(0.80, 0.25, 0.76, 0.15, 'measured state')
arrow(0.58, 0.15, 0.38, 0.35, 'error signals')
arrow(0.58, 0.15, 0.28, 0.68, 'VS feedback')

# Supervisor influence
arrow(0.15, 0.26, 0.20, 0.35, 'override/cap', color='#8b3a2e')
arrow(0.15, 0.26, 0.58, 0.54, None, color='#8b3a2e')

ax.text(0.5, 0.95, 'IFC VTOL Control Architecture (diagnostic view)', ha='center', va='center', fontsize=14, weight='bold')
ax.text(0.5, 0.91,
        'Use this map to relate log columns to control blocks when diagnosing instability.',
        ha='center', va='center', fontsize=10, color='#344054')

plt.tight_layout()
plt.show()


In [ ]:
def list_ifc_logs(logs_dir: Path):
    files = sorted(logs_dir.glob('ifc_log_*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
    rows = []
    for p in files:
        st = p.stat()
        rows.append({
            'name': p.name,
            'path': str(p),
            'size_kb': round(st.st_size / 1024, 1),
            'modified': pd.to_datetime(st.st_mtime, unit='s')
        })
    return pd.DataFrame(rows)

logs_df = list_ifc_logs(LOGS_DIR)
display(logs_df.head(12))

if logs_df.empty:
    raise RuntimeError('No ifc_log_*.csv files found.')

# Change this to analyze a specific file, or keep None for latest.
SELECTED_LOG = None
selected_path = Path(logs_df.iloc[0]['path']) if SELECTED_LOG is None else (LOGS_DIR / SELECTED_LOG)
print(f"Selected log: {selected_path.name}")


In [ ]:
STAGE_RE = re.compile(r'^#\\s*att_stage=([A-Z_]+)\\s+t_(?:air_)?s=([0-9.]+)')

BOOL_COLS = [
    'vtol_upset', 'vtol_thr_guard', 'vtol_level_active', 'vtol_truly_airborne',
    'vtol_is_grounded', 'vtol_alloc_alpha_limited', 'em_starving'
]

def parse_ifc_log(path: Path):
    text = path.read_text(encoding='utf-8', errors='replace').splitlines()
    comment_lines = [ln for ln in text if ln.startswith('#')]
    data_lines = [ln for ln in text if (ln and not ln.startswith('#'))]
    if not data_lines:
        raise ValueError(f'No data rows in {path}')

    stage_rows = []
    for ln in comment_lines:
        m = STAGE_RE.match(ln.strip())
        if m:
            stage_rows.append({'stage': m.group(1), 't_stage_s': float(m.group(2)), 'raw': ln})
    stages = pd.DataFrame(stage_rows)

    csv_text = "\n".join(data_lines)
    df = pd.read_csv(io.StringIO(csv_text))

    for c in BOOL_COLS:
        if c in df.columns:
            df[c] = df[c].astype(str).str.lower().map({'true': True, 'false': False})

    for c in df.columns:
        if c in BOOL_COLS:
            continue
        if c == 'ship_status':
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            continue
        try:
            df[c] = pd.to_numeric(df[c], errors='raise')
        except Exception:
            pass

    meta = {
        'path': str(path),
        'name': path.name,
        'n_rows': len(df),
        'comment_count': len(comment_lines),
        'stages': stages
    }
    return df, meta

df, meta = parse_ifc_log(selected_path)
print(meta['name'], 'rows=', meta['n_rows'])
display(meta['stages'])
display(df.head(3))


In [ ]:
REQUIRED_COLUMNS = [
    't_s', 'pitch_deg', 'bank_deg', 'pitch_rate_degs', 'roll_rate_degs',
    'pitch_cmd', 'roll_cmd', 'vtol_upset', 'vtol_level_active',
    'alloc_alpha', 'vtol_cmd_pitch_postcap', 'vtol_cmd_roll_postcap'
]

print('Notebook Health Check')
print('- selected_path:', selected_path)
print('- dataframe shape:', df.shape)

present = [c for c in REQUIRED_COLUMNS if c in df.columns]
missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
print(f'- required columns present: {len(present)}/{len(REQUIRED_COLUMNS)}')
if missing:
    print('- missing columns:', ', '.join(missing))
    print('  FIX: verify this is an IFC vtol_test log with full telemetry columns.')
else:
    print('- missing columns: none')

# Diagram render sanity checks.
render_checks = {
    'matplotlib_imported': 'plt' in globals(),
    'patches_imported': 'patches' in globals(),
    'block_function_defined': 'block' in globals(),
    'arrow_function_defined': 'arrow' in globals(),
    'diagram_labels_defined': 'DIAGRAM_LABELS' in globals(),
}
for k, ok in render_checks.items():
    print(f"- {k}: {'OK' if ok else 'MISSING'}")

if not all(render_checks.values()):
    print('FIX: run the Control System Diagram code cell before plotting sections.')

if df.empty:
    print('FIX: parsed dataframe is empty. Check selected log path and CSV parser cell.')
else:
    tmin = float(df['t_s'].min()) if 't_s' in df.columns else float('nan')
    tmax = float(df['t_s'].max()) if 't_s' in df.columns else float('nan')
    print(f'- t_s range: {tmin:.3f} .. {tmax:.3f}')


In [ ]:
def first_row(df, cond):
    m = cond(df)
    return df.loc[m].head(1)

def event_summary(df):
    out = {}
    out['first_airborne'] = first_row(
        df,
        lambda x: x.get('vtol_truly_airborne', pd.Series(False, index=x.index)).fillna(False),
    )
    out['first_level_active'] = first_row(
        df,
        lambda x: x.get('vtol_level_active', pd.Series(False, index=x.index)).fillna(False),
    )
    out['first_upset'] = first_row(
        df,
        lambda x: x.get('vtol_upset', pd.Series(False, index=x.index)).fillna(False),
    )

    if 'vtol_cmd_pitch_postcap' in df:
        pcap = df['vtol_cmd_pitch_postcap'].abs()
        pth = max(0.3, float(pcap.quantile(0.99)) * 0.95)
        out['first_pitch_cap'] = first_row(df, lambda x: x['vtol_cmd_pitch_postcap'].abs() >= pth)
        out['pitch_cap_threshold'] = pth

    if 'vtol_cmd_roll_postcap' in df:
        rcap = df['vtol_cmd_roll_postcap'].abs()
        rth = max(0.3, float(rcap.quantile(0.99)) * 0.95)
        out['first_roll_cap'] = first_row(df, lambda x: x['vtol_cmd_roll_postcap'].abs() >= rth)
        out['roll_cap_threshold'] = rth

    return out

def row_to_record(event_name, row):
    return {
        'event': event_name,
        't_s': row.get('t_s', np.nan),
        'alt_agl_m': row.get('alt_agl_m', np.nan),
        'pitch_deg': row.get('pitch_deg', np.nan),
        'bank_deg': row.get('bank_deg', np.nan),
        'pitch_rate_degs': row.get('pitch_rate_degs', np.nan),
        'roll_rate_degs': row.get('roll_rate_degs', np.nan),
        'pitch_cmd': row.get('pitch_cmd', np.nan),
        'roll_cmd': row.get('roll_cmd', np.nan),
        'ship_status': row.get('ship_status', ''),
    }

ev = event_summary(df)
rows = []
for k in ['first_airborne', 'first_level_active', 'first_pitch_cap', 'first_roll_cap', 'first_upset']:
    v = ev.get(k, pd.DataFrame())
    if isinstance(v, pd.DataFrame) and not v.empty:
        rows.append(row_to_record(k, v.iloc[0]))

events_df = pd.DataFrame(rows)
print('Event Timeline')
display(events_df)

print(f"pitch cap threshold ~ {ev.get('pitch_cap_threshold', np.nan):.3f}")
print(f"roll cap threshold  ~ {ev.get('roll_cap_threshold', np.nan):.3f}")

if not events_df.empty:
    msg = []
    if (events_df['event'] == 'first_pitch_cap').any() and (events_df['event'] == 'first_upset').any():
        tp = float(events_df.loc[events_df['event'] == 'first_pitch_cap', 't_s'].iloc[0])
        tu = float(events_df.loc[events_df['event'] == 'first_upset', 't_s'].iloc[0])
        msg.append(f"Pitch saturation leads upset by {tu - tp:.2f}s")
    if (events_df['event'] == 'first_roll_cap').any() and (events_df['event'] == 'first_upset').any():
        tr = float(events_df.loc[events_df['event'] == 'first_roll_cap', 't_s'].iloc[0])
        tu = float(events_df.loc[events_df['event'] == 'first_upset', 't_s'].iloc[0])
        msg.append(f"Roll saturation leads upset by {tu - tr:.2f}s")
    if msg:
        print('Interpretation:')
        for m in msg:
            print('-', m)


In [ ]:
def add_stage_lines(ax, stages_df):
    if stages_df is None or stages_df.empty:
        return
    for _, r in stages_df.iterrows():
        t = r['t_stage_s']
        s = r['stage']
        ax.axvline(t, color='gray', ls='--', alpha=0.5)
        ax.text(t, 0.98, s, transform=ax.get_xaxis_transform(), va='top', ha='left', fontsize=8)

def plot_if_exists(ax, x, df, col, **kwargs):
    if col in df.columns:
        ax.plot(x, df[col], label=col, **kwargs)

stages = meta['stages']
t = df['t_s']

fig, axs = plt.subplots(4, 1, figsize=(15, 15), sharex=True)

plot_if_exists(axs[0], t, df, 'pitch_deg', color='tab:blue')
plot_if_exists(axs[0], t, df, 'bank_deg', color='tab:orange')
axs[0].set_ylabel('deg')
axs[0].set_title('Attitude (degrees)')
axs[0].legend(loc='upper right')
add_stage_lines(axs[0], stages)

plot_if_exists(axs[1], t, df, 'pitch_rate_degs', color='tab:blue')
plot_if_exists(axs[1], t, df, 'roll_rate_degs', color='tab:orange')
axs[1].set_ylabel('deg/s')
axs[1].set_title('Angular Rates (deg/s)')
axs[1].legend(loc='upper right')
add_stage_lines(axs[1], stages)

plot_if_exists(axs[2], t, df, 'pitch_cmd', color='tab:blue')
plot_if_exists(axs[2], t, df, 'roll_cmd', color='tab:orange')
plot_if_exists(axs[2], t, df, 'vtol_cmd_pitch_postcap', ls='--', alpha=0.8, color='tab:blue')
plot_if_exists(axs[2], t, df, 'vtol_cmd_roll_postcap', ls='--', alpha=0.8, color='tab:orange')
axs[2].axhline(0, color='black', lw=0.8, alpha=0.5)
axs[2].set_ylabel('command')
axs[2].set_title('Controller Commands (solid=final cmd, dashed=post-cap)')
axs[2].legend(loc='upper right', ncol=2)
add_stage_lines(axs[2], stages)

plot_if_exists(axs[3], t, df, 'alt_agl_m', color='tab:green')
plot_if_exists(axs[3], t, df, 'vs_ms', color='tab:red')
plot_if_exists(axs[3], t, df, 'collective', color='tab:purple')
axs[3].set_ylabel('mixed units')
axs[3].set_xlabel('t_s')
axs[3].set_title('Altitude / Vertical Speed / Collective')
axs[3].legend(loc='upper right')
add_stage_lines(axs[3], stages)

if 'vtol_upset' in df.columns:
    upset_t = df.loc[df['vtol_upset'].fillna(False), 't_s']
    for ax in axs:
        for ut in upset_t:
            ax.axvline(ut, color='red', alpha=0.08)

plt.tight_layout()
plt.show()

print('How to read this panel:')
print('- Red vertical tint = upset active.')
print('- Look for divergence in rates before attitude departs.')
print('- If commands rail (flat near limits), controller is authority-limited or over-driving gains.')


## Setpoints and Errors (Controller Intent)

This section makes the control intent explicit by plotting targets vs measurements.

What is being tracked:
- **Pitch attitude target**: `0 deg` (nose level) when auto-level is active.
- **Bank attitude target**: `0 deg` (wings level) when auto-level is active.
- **Angular-rate targets**: `0 deg/s` (damping target).
- **Vertical-speed target**: `vs_cmd_ms` (from throttle/altitude logic).

Notes:
- During manual stick input or some upset conditions, effective targeting can shift due to overrides/caps.
- `vtol_pitch_err` and `vtol_roll_err` are the authoritative loop errors; use these for exact controller state.


In [ ]:
df_view = df.copy()

# Nominal setpoints used by level-hold logic.
df_view['pitch_setpoint_deg'] = 0.0
df_view['bank_setpoint_deg'] = 0.0
df_view['pitch_rate_setpoint_degs'] = 0.0
df_view['roll_rate_setpoint_degs'] = 0.0

# Derived tracking errors in measurement space (for visualization).
if 'pitch_deg' in df_view.columns:
    df_view['pitch_error_deg_vis'] = df_view['pitch_setpoint_deg'] - df_view['pitch_deg']
if 'bank_deg' in df_view.columns:
    df_view['bank_error_deg_vis'] = df_view['bank_setpoint_deg'] - df_view['bank_deg']
if 'vs_cmd_ms' in df_view.columns and 'vs_ms' in df_view.columns:
    df_view['vs_error_ms_vis'] = df_view['vs_cmd_ms'] - df_view['vs_ms']

fig, axs = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

t = df_view['t_s']

# Attitude setpoint vs measured.
if 'pitch_deg' in df_view.columns:
    axs[0].plot(t, df_view['pitch_deg'], label='pitch_deg (measured)', color='tab:blue')
axs[0].plot(t, df_view['pitch_setpoint_deg'], '--', label='pitch_setpoint_deg', color='tab:blue', alpha=0.6)
if 'bank_deg' in df_view.columns:
    axs[0].plot(t, df_view['bank_deg'], label='bank_deg (measured)', color='tab:orange')
axs[0].plot(t, df_view['bank_setpoint_deg'], '--', label='bank_setpoint_deg', color='tab:orange', alpha=0.6)
axs[0].set_ylabel('deg')
axs[0].set_title('Attitude Targets vs Measured Attitude')
axs[0].legend(loc='upper right', ncol=2)
add_stage_lines(axs[0], stages)

# Error traces.
if 'pitch_error_deg_vis' in df_view.columns:
    axs[1].plot(t, df_view['pitch_error_deg_vis'], label='pitch_error_deg_vis = 0 - pitch_deg', color='tab:blue')
if 'bank_error_deg_vis' in df_view.columns:
    axs[1].plot(t, df_view['bank_error_deg_vis'], label='bank_error_deg_vis = 0 - bank_deg', color='tab:orange')
if 'vtol_pitch_err' in df_view.columns:
    axs[1].plot(t, df_view['vtol_pitch_err'], '--', label='vtol_pitch_err (logged)', color='tab:blue', alpha=0.7)
if 'vtol_roll_err' in df_view.columns:
    axs[1].plot(t, df_view['vtol_roll_err'], '--', label='vtol_roll_err (logged)', color='tab:orange', alpha=0.7)
axs[1].axhline(0, color='black', lw=0.8, alpha=0.5)
axs[1].set_ylabel('error')
axs[1].set_title('Error Signals (visual vs logged loop errors)')
axs[1].legend(loc='upper right', ncol=2)
add_stage_lines(axs[1], stages)

# Rate/VS targets.
if 'pitch_rate_degs' in df_view.columns:
    axs[2].plot(t, df_view['pitch_rate_degs'], label='pitch_rate_degs (measured)', color='tab:blue')
if 'roll_rate_degs' in df_view.columns:
    axs[2].plot(t, df_view['roll_rate_degs'], label='roll_rate_degs (measured)', color='tab:orange')
if 'vs_ms' in df_view.columns:
    axs[2].plot(t, df_view['vs_ms'], label='vs_ms (measured)', color='tab:green')
if 'vs_cmd_ms' in df_view.columns:
    axs[2].plot(t, df_view['vs_cmd_ms'], '--', label='vs_cmd_ms (setpoint)', color='tab:green', alpha=0.7)
axs[2].axhline(0, color='black', lw=0.8, alpha=0.5)
axs[2].set_ylabel('deg/s or m/s')
axs[2].set_xlabel('t_s')
axs[2].set_title('Rate and Vertical-Speed Targets vs Measured')
axs[2].legend(loc='upper right', ncol=2)
add_stage_lines(axs[2], stages)

if 'vtol_upset' in df_view.columns:
    upset_t = df_view.loc[df_view['vtol_upset'].fillna(False), 't_s']
    for ax in axs:
        for ut in upset_t:
            ax.axvline(ut, color='red', alpha=0.08)

plt.tight_layout()
plt.show()

print('How to read this panel:')
print('- If measured traces diverge from dashed setpoint traces, tracking is poor.')
print('- Compare *_error_vis to vtol_*_err to understand sign conventions and controller internal error.')
print('- Large sustained nonzero error with railed commands indicates authority or stability limits.')


In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

for c in ['vtol_pitch_p', 'vtol_pitch_i', 'vtol_pitch_d', 'vtol_pitch_unsat', 'vtol_cmd_pitch_postcap']:
    if c in df.columns:
        style = '--' if 'unsat' in c else ':' if 'postcap' in c else '-'
        axs[0].plot(df['t_s'], df[c], style, label=c)
axs[0].axhline(0, color='black', lw=0.8, alpha=0.5)
axs[0].set_title('Pitch Controller Terms')
axs[0].set_ylabel('command contribution')
axs[0].legend(loc='upper right', ncol=3)
add_stage_lines(axs[0], stages)

for c in ['vtol_roll_p', 'vtol_roll_i', 'vtol_roll_d', 'vtol_roll_unsat', 'vtol_cmd_roll_postcap']:
    if c in df.columns:
        style = '--' if 'unsat' in c else ':' if 'postcap' in c else '-'
        axs[1].plot(df['t_s'], df[c], style, label=c)
axs[1].axhline(0, color='black', lw=0.8, alpha=0.5)
axs[1].set_title('Roll Controller Terms')
axs[1].set_ylabel('command contribution')
axs[1].set_xlabel('t_s')
axs[1].legend(loc='upper right', ncol=3)
add_stage_lines(axs[1], stages)

plt.tight_layout()
plt.show()

print('How to read this panel:')
print('- P-term dominates static attitude error correction.')
print('- D-term should oppose rate growth (damping).')
print('- Unsat vs postcap gap indicates hard-cap clipping.')


In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(15, 11), sharex=True)

for c in ['vtol_diff_raw', 'vtol_diff_attn', 'vtol_diff_upset']:
    if c in df.columns:
        axs[0].plot(df['t_s'], df[c], label=c)
axs[0].set_title('Differential Scaling')
axs[0].set_ylabel('scale (0..1)')
axs[0].legend(loc='upper right')
add_stage_lines(axs[0], stages)

for c in ['alloc_alpha', 'vtol_clamp_high_n', 'vtol_clamp_low_n']:
    if c in df.columns:
        axs[1].plot(df['t_s'], df[c], label=c)
axs[1].set_title('Allocator Health')
axs[1].set_ylabel('alpha / clamp count')
axs[1].legend(loc='upper right')
add_stage_lines(axs[1], stages)

eng_cols = [c for c in ['eng1_lim', 'eng2_lim', 'eng3_lim', 'eng4_lim'] if c in df.columns]
for c in eng_cols:
    axs[2].plot(df['t_s'], df[c], label=c)
axs[2].set_title('Per-Engine Limiters')
axs[2].set_ylabel('limit fraction')
axs[2].set_xlabel('t_s')
if eng_cols:
    axs[2].legend(loc='upper right', ncol=4)
add_stage_lines(axs[2], stages)

plt.tight_layout()
plt.show()

print('How to read this panel:')
print('- alloc_alpha < 1 means allocator had to scale differential authority.')
print('- clamp_high/low spikes mean engines are pinning at limiter bounds.')
print('- Persistent clamp + railing commands usually means control authority bottleneck.')


In [ ]:
def airborne_time(df):
    if 'vtol_truly_airborne' not in df.columns or 't_s' not in df.columns:
        return df['t_s']
    idx = df.index[df['vtol_truly_airborne'].fillna(False)]
    if len(idx) == 0:
        return df['t_s']
    t0 = df.loc[idx[0], 't_s']
    return df['t_s'] - t0

N_COMPARE = 6
compare_logs = [Path(p) for p in logs_df['path'].head(N_COMPARE)]
runs = []
for p in compare_logs:
    dfi, _ = parse_ifc_log(p)
    dfi = dfi.copy()
    dfi['t_air_s'] = airborne_time(dfi)
    dfi['run'] = p.name
    runs.append(dfi)

big = pd.concat(runs, ignore_index=True)

fig, axs = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
for run_name, g in big.groupby('run'):
    if 'pitch_deg' in g.columns:
        axs[0].plot(g['t_air_s'], g['pitch_deg'], label=run_name)
    if 'bank_deg' in g.columns:
        axs[1].plot(g['t_air_s'], g['bank_deg'], label=run_name)

axs[0].set_title('Pitch vs Airborne Time (recent runs)')
axs[0].set_ylabel('pitch_deg')
axs[0].legend(loc='upper left', fontsize=8)
axs[1].set_title('Bank vs Airborne Time (recent runs)')
axs[1].set_ylabel('bank_deg')
axs[1].set_xlabel('t_air_s')

plt.tight_layout()
plt.show()

print('How to read this panel:')
print('- This aligns runs at first airborne timestamp.')
print('- Earlier divergence = lower robustness.')
print('- Use this view to verify whether each code change improved stability window.')


## Glossary and Interpretation Guide

| Term | Plain-English meaning | Where to see it in logs | Why it matters |
|---|---|---|---|
| Allocator | The stage that converts roll/pitch/collective commands into per-engine limiter values. | `alloc_alpha`, `vtol_alloc_alpha_limited`, `eng*_lim` | Tells you if requested control was feasible. |
| `alloc_alpha` | Differential scale actually applied by allocator (1 = full requested differential). | `alloc_alpha` | `<1` means commanded differential had to be reduced. |
| Setpoint | Target value the controller is trying to track. | `vs_cmd_ms`, implied 0-deg attitude targets | Needed to interpret whether tracking is correct. |
| Error | Difference between target and measured signal in controller sign convention. | `vtol_pitch_err`, `vtol_roll_err`, visual `*_error_deg_vis` | Error sign drives command direction; wrong sign can destabilize loop. |
| Saturation | Controller output hitting configured hard caps. | `vtol_cmd_*_precap` vs `vtol_cmd_*_postcap` | Sustained saturation often precedes divergence. |
| Command cap (`postcap`) | Command after hard limit is applied. | `vtol_cmd_pitch_postcap`, `vtol_cmd_roll_postcap` | Shows what actually enters mixer after limiting. |
| Clamp high/low | Number of engines pinned at upper/lower limiter bounds. | `vtol_clamp_high_n`, `vtol_clamp_low_n` | Persistent clamps indicate authority bottleneck. |
| Upset mode | Safety override mode activated by large attitude/rate excursions. | `vtol_upset`, `vtol_cmd_*_postupset` | Confirms when supervisor takes over from normal loop. |
| Differential authority | Amount of roll/pitch differential allowed through mixer. | `vtol_diff_raw`, `vtol_diff_attn`, `vtol_diff_upset` | Low differential limits stabilization torque. |
| Collective | Common-mode thrust request shared across VTOL engines. | `collective`, `hover_coll` | Drives lift/vertical behavior separate from differential torque. |
| Cross-axis coupling | Pitch correction causing roll response (or reverse) due to shared actuators. | Compare `pitch_cmd` with `bank_deg`/`roll_rate_degs` and vice versa | Main cause of coupled departures in MIMO VTOL control. |
| Anti-windup | Integrator protection when output is saturated/authority-limited. | `vtol_*_i`, `vtol_*_unsat`, `postcap` | Prevents integrator from accumulating unusable command. |
| P / I / D terms | Proportional / Integral / Derivative command contributions. | `vtol_pitch_p/i/d`, `vtol_roll_p/i/d` | Shows which term dominates or destabilizes loop. |
| Unsaturated output (`*_unsat`) | Raw controller sum before hard cap. | `vtol_pitch_unsat`, `vtol_roll_unsat` | Gap to `postcap` quantifies clipping severity. |
| Phase lag | Delay between measured error and effective corrective action. | Stage transitions + cmd/rate timing in plots | Excess lag can turn damping into oscillation. |

### Common Symptoms
- **Rate-led divergence**: rates rise first, then attitude departs.
- **Command rail**: command sticks near cap (`|cmd|` ~ cap) for sustained periods.
- **Allocator bottleneck**: `alloc_alpha` drops and/or clamp counts rise while errors keep growing.
- **Upset churn**: repeated transitions in/out of upset with alternating command sign.

### Practical Tuning Order
1. Ensure sign conventions are correct (`P` and `D` oppose error/rate in command space).
2. Tune `D` first for damping (avoid oscillation), then `P` for stiffness.
3. Add only small `I` after PD is stable.
4. Verify caps/upset limits still allow enough recovery authority.
5. Re-check multi-run comparison for consistency, not just one lucky run.
